# Getting the Bronze Data From External storage

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
import builtins
from pyspark.sql.functions import current_date, lit
import builtins

base_path = "abfss://orderreviews@adlsstdout001tgt.dfs.core.windows.net/Reviews/"
# ---- Year ----
folders = dbutils.fs.ls(base_path)
latest_year = builtins.max([f.name.replace("/", "") for f in folders])

# ---- Month ----
folders = dbutils.fs.ls(f"{base_path}{latest_year}/")
latest_month = builtins.max([f.name.replace("/", "") for f in folders])

# ---- Day ----
folders = dbutils.fs.ls(f"{base_path}{latest_year}/{latest_month}/")
latest_day = builtins.max([f.name.replace("/", "") for f in folders])

# ---- Final Path ----
latest_path = f"{base_path}{latest_year}/{latest_month}/{latest_day}/"

print(f"Reading from: {latest_path}")

# ---- File Date ----
file_date = f"{latest_year}-{latest_month}-{latest_day}"

# ---- Read parquet ----
schema = StructType([

    StructField("reviewid", StringType(), True),
    StructField("orderid", StringType(), True),
    StructField("reviewscore", StringType(), True),
    StructField("reviewcommenttitle", StringType(), True),
    StructField("reviewcommentmessage", StringType(), True),
    StructField("reviewcreationdate", StringType(), True),
    StructField("reviewanswertimestamp", StringType(), True),
    StructField("ingest_date", StringType(), True),
    StructField("file_date", StringType(), True)

])

raw_df = (
    spark.read
    .option("recursiveFileLookup", "true")
    .schema(schema)
    .parquet(base_path)
    .withColumn("ingest_date", F.current_date())
    .withColumn("file_date", F.lit(file_date))
)

display(raw_df.limit(10))

# Data Cleaning

In [0]:
import builtins
from pyspark.sql.functions import current_date, lit
import builtins
from pyspark.sql.functions import current_date, lit
from pyspark.sql.window import Window
from pyspark.sql.functions import col
from pyspark.sql.functions import *
from pyspark.sql.types import *


df = raw_df

# Remove duplicates
df = df.dropDuplicates(["reviewid"])

# Trim string columns
string_cols = [c for c, t in df.dtypes if t == "string"]

for c in string_cols:
    df = df.withColumn(c, trim(col(c)))



# Fill null comments
df = df.fillna({
    "reviewcommenttitle": "No Title",
    "reviewcommentmessage": "No Comment"
})

# Add sentiment column
df = df.withColumn(
    "review_sentiment",
    when(col("reviewscore") >= 4, "POSITIVE")
    .when(col("reviewscore") == 3, "NEUTRAL")
    .otherwise("NEGATIVE")
)

from pyspark.sql.functions import *

##df = df.withColumn(
   ##"datecond",
   ##when(
      ##  col("reviewcreationdate") != to_date(col("reviewcreationdate")),
       ## None
    ##).otherwise(col("reviewcreationdate"))
##)

df.createOrReplaceTempView("reviews")

filter_df = spark.sql("""
    SELECT *
    FROM reviews
    WHERE  reviewcreationdate  != "f63f9a7699e3674c80a4ba92e56dfbb8"
""")

df = filter_df \
    .withColumn(
        "created_timestamp",
        F.current_timestamp()
    ) \
    .withColumn(
        "updated_timestamp",
        F.current_timestamp()
    ) \
    .withColumn(
        "source_system",
        F.lit("ecommerce")
    )\
    .withColumn(
        "reviewcreationdate",
        F.to_timestamp(
            F.col("reviewcreationdate"),
            "yyyy-MM-dd HH:mm:ss"
        )
    )\
    .withColumn(
        "reviewanswertimestamp",
        F.to_timestamp(
            F.col("reviewanswertimestamp"),
            "yyyy-MM-dd HH:mm:ss"
        )
    )

final_df = df.select(
    "reviewid",
    "orderid",

    "reviewscore",
    "reviewcommenttitle",
    "reviewcommentmessage",
    "review_sentiment",

    "reviewcreationdate",
    "reviewanswertimestamp",

    "source_system",
    "file_date",
    "ingest_date",

    "created_timestamp",
    "updated_timestamp"
)

display(final_df)

In [0]:
tablename = "ecom.silver.fact_silver_orderreviews"

In [0]:
final_df.printSchema()

In [0]:
(
        final_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .option(
            "path",
            "abfss://silver@silverprocessdbstorage.dfs.core.windows.net/OrderReviews"
        )
        .saveAsTable(tablename)
    )

print("Full Load Completed Successfully")